# AI Morph Ads — Colab / Kaggle runner

One product image → four ad creatives (Luxury / Minimal / Urban / Nostalgic).

**Runtime:** GPU (T4 is enough — 15 GB VRAM).

This notebook:
1. Clones ComfyUI + required custom nodes
2. Clones *this* project and installs Python deps
3. Downloads all model weights into `ComfyUI/models/...`
4. Starts ComfyUI as a background server
5. Runs `scripts/run_pipeline.py` for a product image you upload

> **Note on LoRAs.** Style LoRAs from CivitAI are not auto-downloaded (their URLs rotate and often need an API key). Cell 4 prints the 4 filenames expected under `ComfyUI/models/loras/` — upload those manually before running the pipeline, or edit `configs/variants.yaml` to match the LoRAs you already have.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone ComfyUI + custom nodes + this project

In [ ]:
import os, sys, subprocess, pathlib

WORK = pathlib.Path('/content') if pathlib.Path('/content').exists() else pathlib.Path('/kaggle/working')
os.chdir(WORK)
print('Working dir:', WORK)

if not (WORK / 'ComfyUI').exists():
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git

NODES = WORK / 'ComfyUI' / 'custom_nodes'
NODES.mkdir(parents=True, exist_ok=True)

for name, repo in [
    ('ComfyUI-Manager',           'https://github.com/ltdrdata/ComfyUI-Manager.git'),
    ('ComfyUI_IPAdapter_plus',    'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git'),
    ('comfyui_controlnet_aux',    'https://github.com/Fannovel16/comfyui_controlnet_aux.git'),
]:
    target = NODES / name
    if not target.exists():
        print(f'git clone {name}')
        subprocess.run(['git', 'clone', '--depth', '1', repo, str(target)], check=True)

print('\nInstalling ComfyUI requirements...')
!pip install -q -r ComfyUI/requirements.txt

print('Installing custom-node requirements...')
for name in ['ComfyUI-Manager', 'ComfyUI_IPAdapter_plus', 'comfyui_controlnet_aux']:
    req = NODES / name / 'requirements.txt'
    if req.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=True)

In [ ]:
# Clone or upload this project.
# Option A: if you've pushed it to your own GitHub, clone it here:
# !git clone https://github.com/<you>/ai-morph-ads.git

# Option B: upload the project folder via the Colab file browser, then:
PROJECT = WORK / 'AI Morph Ads'
assert PROJECT.exists(), f'Upload / clone the project to {PROJECT}'

!pip install -q -r "$PROJECT/requirements.txt"

## 3. Download model weights

~10 GB total with SDXL. First run takes 5–10 min on a Colab T4; subsequent sessions skip anything already on disk.

In [ ]:
!python "$PROJECT/scripts/download_models.py" --comfy-root "$WORK/ComfyUI"

## 4. Upload style LoRAs + product image

Use the Colab file browser (left sidebar) to drop:
- 4 `.safetensors` LoRAs into `ComfyUI/models/loras/` (filenames must match `configs/variants.yaml`)
- 1 product image (PNG/JPG) into `AI Morph Ads/inputs/`

In [ ]:
loras = sorted((WORK / 'ComfyUI' / 'models' / 'loras').glob('*.safetensors'))
print('LoRAs found:')
for p in loras:
    print(' -', p.name)

inputs = sorted((PROJECT / 'inputs').glob('*'))
print('\nProduct images found:')
for p in inputs:
    if p.is_file():
        print(' -', p.name)

## 5. Start ComfyUI as a background server

We launch it with `--listen 127.0.0.1 --port 8188` and `--medvram` so it fits inside T4's 15 GB budget (SDXL base is loaded in stage B only).

In [ ]:
import subprocess, time, requests

LOG = WORK / 'comfyui.log'
log_f = open(LOG, 'w')
proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188', '--medvram'],
    cwd=str(WORK / 'ComfyUI'),
    stdout=log_f, stderr=subprocess.STDOUT,
)
print('ComfyUI PID:', proc.pid)

# Wait until /system_stats responds (server up)
for i in range(60):
    try:
        r = requests.get('http://127.0.0.1:8188/system_stats', timeout=2)
        if r.ok:
            print('ComfyUI is up.')
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError(f'ComfyUI did not start in time — check {LOG}')

## 6. Run the 4-variant pipeline

In [ ]:
PRODUCT_FILENAME = 'bottle.png'      # <- your uploaded product image in inputs/
BRAND            = 'NOIR'
TAGLINE          = 'Timeless.'
PRODUCT_NAME     = 'perfume bottle'  # generic noun used inside the prompt

!python "$PROJECT/scripts/run_pipeline.py" \
    --product "$PROJECT/inputs/$PRODUCT_FILENAME" \
    --brand "$BRAND" \
    --tagline "$TAGLINE" \
    --product-name "$PRODUCT_NAME" \
    --server http://127.0.0.1:8188

## 7. Preview the 4 creatives

In [ ]:
from IPython.display import Image, display
for p in sorted((PROJECT / 'outputs').glob('*.png')):
    if p.name.endswith('_raw.png'):
        continue
    print(p.name)
    display(Image(str(p), width=512))